# Parent Document Retriever — Small Chunks, Full Context
## A Hands-On Workshop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/71-parent-document-retriever/parent_document_retriever_workbook.ipynb)

Embeds small child chunks for precise retrieval but returns the full parent document as context. Solves the classic RAG precision-vs-context tradeoff: small chunks match queries well, large parents answer questions well.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Precision-vs-context tradeoff; child chunks indexed, parent docs returned |
| 2 | **Setup** — CHILD_CHUNK_SIZE=100, PARENT_CHUNK_SIZE=600; Chroma + InMemoryStore |
| 3 | **Retrieve** — Small chunk matches query; retriever fetches the full parent |
| 4 | **Child vs Parent** — Compare raw similarity_search results vs ParentDocumentRetriever |
| 5 | **Full Pipeline** — START → retrieve_parent → generate → END |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langchain-chroma", "langchain-text-splitters", "chromadb", "langgraph", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

In [ ]:
from typing import TypedDict
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import END, START, StateGraph

CHILD_CHUNK_SIZE = 100
PARENT_CHUNK_SIZE = 600

SAMPLE_DOCS = [
    "The transformer architecture revolutionized natural language processing. Introduced in 2017 by Vaswani et al., it replaced recurrent networks with self-attention mechanisms. This allowed parallel computation and better capture of long-range dependencies. The key innovation was multi-head attention, which lets the model focus on different parts of the input simultaneously. Transformers became the foundation for BERT, GPT, and all modern large language models. Their ability to scale with more data and computation made them the dominant architecture in AI today.",
    "Retrieval-augmented generation (RAG) combines the strengths of retrieval systems and generative models. Instead of relying solely on parametric knowledge stored in model weights, RAG retrieves relevant documents from an external knowledge base at inference time. This approach reduces hallucination and allows models to access up-to-date information. The typical RAG pipeline has three stages: document indexing, retrieval, and generation. Vector embeddings enable semantic search, where queries and documents are matched by meaning rather than exact keywords.",
    "Neural network training involves adjusting millions of parameters to minimize a loss function. The primary algorithm is stochastic gradient descent (SGD) with backpropagation. In each training iteration, the model makes a forward pass to compute predictions, then a backward pass to compute gradients. The optimizer updates weights based on those gradients. Modern optimizers like Adam adapt the learning rate for each parameter. Regularization techniques like dropout prevent overfitting.",
]

SAMPLE_QUESTIONS = [
    "What was the key innovation in the transformer architecture?",
    "How does RAG reduce hallucination?",
    "What is stochastic gradient descent?",
]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=CHILD_CHUNK_SIZE, chunk_overlap=10)
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=PARENT_CHUNK_SIZE, chunk_overlap=50)
vectorstore = Chroma(collection_name="parent-doc-children", embedding_function=embeddings)
docstore = InMemoryStore()
retriever = ParentDocumentRetriever(vectorstore=vectorstore, docstore=docstore,
    child_splitter=child_splitter, parent_splitter=parent_splitter)
docs = [Document(page_content=d) for d in SAMPLE_DOCS]
retriever.add_documents(docs)

q = SAMPLE_QUESTIONS[0]
child_docs = vectorstore.similarity_search(q, k=3)
parent_docs = retriever.invoke(q)
print(f"Q: {q}")
print(f"Child chunks matched: {len(child_docs)}  avg {sum(len(d.page_content) for d in child_docs)//len(child_docs)} chars")
print(f"Parent docs returned: {len(parent_docs)}  avg {sum(len(d.page_content) for d in parent_docs)//len(parent_docs)} chars")

class ParentDocState(TypedDict):
    question: str; child_chunks: list; parent_docs: list; answer: str

def retrieve_parent_node(state):
    parents = retriever.invoke(state["question"])
    children = vectorstore.similarity_search(state["question"], k=4)
    return {"parent_docs": [d.page_content for d in parents], "child_chunks": [d.page_content for d in children]}

def generate_node(state):
    context = "\n\n".join(state["parent_docs"])
    answer = llm.invoke([HumanMessage(content=f"Answer using the context:\n\n{context}\n\nQuestion: {state['question']}")]).content.strip()
    return {"answer": answer}

g = StateGraph(ParentDocState)
g.add_node("retrieve_parent", retrieve_parent_node)
g.add_node("generate", generate_node)
g.add_edge(START, "retrieve_parent"); g.add_edge("retrieve_parent", "generate"); g.add_edge("generate", END)
app = g.compile()

for q in SAMPLE_QUESTIONS:
    r = app.invoke({"question": q, "child_chunks": [], "parent_docs": [], "answer": ""})
    print(f"\nQ: {q}")
    print(f"  Parents: {len(r['parent_docs'])} docs  |  Children: {len(r['child_chunks'])} chunks")
    print(f"  A: {r['answer'][:200]}")

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*